In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

BASE = Path(r"C:/Users/janku/Documents/KCL/Research Project/Research Project")
REPR_DIR = BASE / "results" / "metrics" / "repr_learn"
KINTSUGI_DIR = BASE / "results" / "metrics" / "kintsugi_health"
FIG_DIR = BASE / "results" / "figures" / "reprlearn_and_kintsugi"
FIG_DIR.mkdir(parents=True, exist_ok=True)

METRICS = ["accuracy", "f1", "roc_auc"]
METRIC_LABELS = {"accuracy": "Accuracy", "f1": "F1", "roc_auc": "ROC-AUC"}


def save_fig(fig, name: str, dpi: int = 200) -> Path:
    path = FIG_DIR / name
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")
    return path


def _split_dataset_model(stem: str, suffix: str) -> tuple[str, str]:
    base = stem[: -len(suffix)]
    if "_" not in base:
        return "unknown", base
    dataset, model = base.split("_", 1)
    return dataset, model


def load_repr_learn(metrics_dir: Path) -> dict[str, pd.DataFrame]:
    cv_summary_rows: list[pd.DataFrame] = []
    cv_fold_frames: list[pd.DataFrame] = []
    dl_summary_rows: list[pd.DataFrame] = []
    dl_pred_frames: list[pd.DataFrame] = []

    if not metrics_dir.exists():
        print(f"Warning: missing metrics dir {metrics_dir}")
        return {
            "cv_summary": pd.DataFrame(),
            "cv_folds": pd.DataFrame(),
            "dl_summary": pd.DataFrame(),
            "dl_predictions": pd.DataFrame(),
        }

    for path in sorted(metrics_dir.glob("*.csv")):
        stem = path.stem
        if stem.startswith("radar_") or stem.endswith("_embeddings"):
            continue
        if stem.endswith("_dl_head_no_folds_summary"):
            dataset, backbone = _split_dataset_model(stem, "_dl_head_no_folds_summary")
            df = pd.read_csv(path)
            df["dataset"] = dataset
            df["backbone"] = backbone
            df["approach"] = "frozen_encoder_dl_head"
            df["model_label"] = f"{dataset} | {backbone} (DL head)"
            dl_summary_rows.append(df)
        elif stem.endswith("_dl_head_no_folds_predictions"):
            dataset, backbone = _split_dataset_model(stem, "_dl_head_no_folds_predictions")
            df = pd.read_csv(path)
            df["dataset"] = dataset
            df["backbone"] = backbone
            df["approach"] = "frozen_encoder_dl_head"
            df["model_label"] = f"{dataset} | {backbone} (DL head)"
            dl_pred_frames.append(df)
        elif stem.endswith("_cv_folds"):
            dataset, backbone = _split_dataset_model(stem, "_cv_folds")
            df = pd.read_csv(path)
            df["dataset"] = dataset
            df["backbone"] = backbone
            df["approach"] = "linear_probe_cv"
            df["model_label"] = f"{dataset} | {backbone} (linear probe)"
            cv_fold_frames.append(df)
        elif stem.endswith("_summary"):
            dataset, backbone = _split_dataset_model(stem, "_summary")
            df = pd.read_csv(path)
            df["dataset"] = dataset
            df["backbone"] = backbone
            df["approach"] = "linear_probe_cv"
            df["model_label"] = f"{dataset} | {backbone} (linear probe)"
            cv_summary_rows.append(df)

    return {
        "cv_summary": pd.concat(cv_summary_rows, ignore_index=True) if cv_summary_rows else pd.DataFrame(),
        "cv_folds": pd.concat(cv_fold_frames, ignore_index=True) if cv_fold_frames else pd.DataFrame(),
        "dl_summary": pd.concat(dl_summary_rows, ignore_index=True) if dl_summary_rows else pd.DataFrame(),
        "dl_predictions": pd.concat(dl_pred_frames, ignore_index=True) if dl_pred_frames else pd.DataFrame(),
    }


def load_kintsugi(metrics_dir: Path) -> dict[str, pd.DataFrame]:
    summary_path = metrics_dir / "dam_whisper_hc_pt_summary.csv"
    pred_path = metrics_dir / "dam_whisper_hc_pt_test_predictions.csv"
    summary = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()
    preds = pd.read_csv(pred_path) if pred_path.exists() else pd.DataFrame()
    if not summary.empty:
        summary["model_label"] = summary["model"]
        summary["approach"] = "dam_whisper"
        summary["dataset"] = "HC/PT raw"
    return {"summary": summary, "predictions": preds}


def cv_summary_long(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        for metric in METRICS:
            mean_col = f"{metric}_mean"
            std_col = f"{metric}_std"
            if mean_col not in row:
                continue
            rows.append(
                {
                    "model_label": row["model_label"],
                    "dataset": row.get("dataset"),
                    "backbone": row.get("backbone"),
                    "approach": row.get("approach"),
                    "site": row.get("site"),
                    "metric": metric,
                    "value": row[mean_col],
                    "std": row.get(std_col, np.nan),
                }
            )
    return pd.DataFrame(rows)


def point_summary_long(df: pd.DataFrame, model_col: str = "model_label") -> pd.DataFrame:
    cols = [c for c in METRICS if c in df.columns]
    if df.empty or not cols:
        return pd.DataFrame()
    id_vars = [c for c in [model_col, "dataset", "backbone", "approach", "site"] if c in df.columns]
    return df.melt(id_vars=id_vars, value_vars=cols, var_name="metric", value_name="value")


repr_data = load_repr_learn(REPR_DIR)
kintsugi = load_kintsugi(KINTSUGI_DIR)

print("Representation learning metrics:")
for key, df in repr_data.items():
    labels = df["model_label"].unique().tolist() if "model_label" in df.columns and not df.empty else []
    print(f"  {key}: {df.shape} — {labels}")

print("\nKintsugi Health metrics:")
for key, df in kintsugi.items():
    print(f"  {key}: {df.shape}")

Representation learning metrics:
  cv_summary: (3, 13) — ['androids | hubert (linear probe)', 'androids | wav2vec2 (linear probe)', 'androids | whisper (linear probe)']
  cv_folds: (15, 11) — ['androids | hubert (linear probe)', 'androids | wav2vec2 (linear probe)', 'androids | whisper (linear probe)']
  dl_summary: (1, 16) — ['androids | wav2vec2 (DL head)']
  dl_predictions: (45, 9) — ['androids | wav2vec2 (DL head)']

Kintsugi Health metrics:
  summary: (1, 15)
  predictions: (26, 8)


In [3]:
# --- 1. Representation learning: GroupKFold linear-probe summary (mean ± std) ---
cv_sum = repr_data["cv_summary"]
if cv_sum.empty:
    print("No linear-probe CV summary files found.")
else:
    long = cv_summary_long(cv_sum)

    g = sns.catplot(
        data=long,
        kind="bar",
        x="backbone",
        y="value",
        hue="metric",
        col="dataset",
        col_wrap=2,
        height=4.5,
        aspect=1.2,
        sharey=False,
    )
    g.set_axis_labels("Backbone", "Score")
    g.set_titles("{col_name}")
    for ax in g.axes.ravel():
        ax.set_ylim(0, 1.05)
    g.fig.suptitle("Representation learning — linear probe (GroupKFold mean)", y=1.03)
    save_fig(g.fig, "01_repr_linear_probe_cv_summary.png")

    # Error-bar version (one metric panel per backbone)
    fig, axes = plt.subplots(1, len(METRICS), figsize=(5 * len(METRICS), 5), squeeze=False)
    for ax, metric in zip(axes.ravel(), METRICS):
        sub = long[long["metric"] == metric].copy()
        order = sorted(sub["backbone"].unique())
        x = np.arange(len(order))
        colors = sns.color_palette("Set2", len(order))
        for i, backbone in enumerate(order):
            row = sub[sub["backbone"] == backbone].iloc[0]
            ax.bar(i, row["value"], yerr=row["std"], capsize=4, color=colors[i], label=backbone)
        ax.set_xticks(x)
        ax.set_xticklabels(order, rotation=20)
        ax.set_ylim(0, 1.05)
        ax.set_title(METRIC_LABELS[metric])
        ax.set_ylabel("Score")
    fig.suptitle("Linear probe CV — mean ± std by backbone", y=1.02)
    fig.tight_layout()
    save_fig(fig, "01_repr_linear_probe_cv_summary_errbars.png")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\01_repr_linear_probe_cv_summary.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\01_repr_linear_probe_cv_summary_errbars.png


In [4]:
# --- 2. Representation learning: per-fold CV curves + heatmaps ---
cv_folds = repr_data["cv_folds"]
if cv_folds.empty:
    print("No CV fold files found.")
else:
    plot_metrics = [m for m in METRICS if m in cv_folds.columns]

    if plot_metrics:
        fig, axes = plt.subplots(1, len(plot_metrics), figsize=(5 * len(plot_metrics), 5), squeeze=False)
        for ax, metric in zip(axes.ravel(), plot_metrics):
            sns.lineplot(
                data=cv_folds,
                x="fold",
                y=metric,
                hue="backbone",
                style="dataset",
                marker="o",
                ax=ax,
            )
            ax.set_title(f"CV {METRIC_LABELS[metric]} by fold")
            ax.set_xlabel("Fold")
            ax.set_xticks(sorted(cv_folds["fold"].dropna().unique()))
            ax.set_ylim(0, 1.05)
            ax.legend(title="", fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
        fig.tight_layout()
        save_fig(fig, "02_repr_linear_probe_cv_folds_lineplots.png")

    for metric in plot_metrics:
        pivot = cv_folds.pivot_table(index="backbone", columns="fold", values=metric, aggfunc="first")
        if pivot.empty:
            continue
        fig, ax = plt.subplots(figsize=(8, max(3.5, 0.6 * len(pivot))))
        sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis", vmin=0, vmax=1, ax=ax)
        ax.set_title(f"Linear probe CV — {METRIC_LABELS[metric]} (backbone × fold)")
        save_fig(fig, f"02_repr_linear_probe_cv_heatmap_{metric}.png")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\02_repr_linear_probe_cv_folds_lineplots.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\02_repr_linear_probe_cv_heatmap_accuracy.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\02_repr_linear_probe_cv_heatmap_f1.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\02_repr_linear_probe_cv_heatmap_roc_auc.png


In [5]:
# --- 3. Representation learning: frozen encoder + DL head (single split) ---
dl_sum = repr_data["dl_summary"]
if dl_sum.empty:
    print("No DL-head summary files found.")
else:
    long = point_summary_long(dl_sum)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=long, x="backbone", y="value", hue="metric", ax=ax)
    ax.set_ylim(0, 1.05)
    ax.set_title("Representation learning — frozen encoder + trainable DL head (held-out test)")
    ax.set_xlabel("Backbone")
    ax.set_ylabel("Score")
    ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_fig(fig, "03_repr_dl_head_test_summary.png")

dl_preds = repr_data["dl_predictions"]
if not dl_preds.empty and "pred_prob" in dl_preds.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(
        data=dl_preds,
        x="pred_prob",
        hue="depressed",
        multiple="stack",
        bins=20,
        ax=ax,
    )
    ax.set_title("DL head — predicted probability distribution (test set)")
    ax.set_xlabel("P(depressed)")
    save_fig(fig, "03_repr_dl_head_pred_prob_hist.png")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\03_repr_dl_head_test_summary.png


In [6]:
# --- 4. Kintsugi Health: DAM-like Whisper on HC/PT raw audio ---
k_sum = kintsugi["summary"]
k_preds = kintsugi["predictions"]

if k_sum.empty:
    print("No Kintsugi summary found — run KintsugiHealth.ipynb cell 1 first.")
else:
    long = point_summary_long(k_sum, model_col="model")
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=long, x="model", y="value", hue="metric", ax=ax)
    ax.set_ylim(0, 1.05)
    ax.set_title("Kintsugi Health — DAM-like Whisper (held-out test)")
    ax.tick_params(axis="x", rotation=15)
    ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_fig(fig, "04_kintsugi_dam_test_summary.png")

if not k_preds.empty and "pred_prob" in k_preds.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(
        data=k_preds,
        x="pred_prob",
        hue="subgroup",
        multiple="stack",
        bins=20,
        ax=ax,
    )
    ax.set_title("Kintsugi DAM — P(depressed) by subgroup (HC vs PT)")
    ax.set_xlabel("P(depressed)")
    save_fig(fig, "04_kintsugi_dam_pred_prob_by_subgroup.png")

    if {"depressed", "pred_label"}.issubset(k_preds.columns):
        cm = pd.crosstab(k_preds["depressed"], k_preds["pred_label"])
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True (depressed)")
        ax.set_title("Kintsugi DAM — confusion matrix (test)")
        save_fig(fig, "04_kintsugi_dam_confusion_matrix.png")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\04_kintsugi_dam_test_summary.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\04_kintsugi_dam_pred_prob_by_subgroup.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\04_kintsugi_dam_confusion_matrix.png


In [7]:
# --- 5. Combined overview: all available held-out / CV-mean metrics ---
combined_rows = []

if not repr_data["cv_summary"].empty:
    for _, row in repr_data["cv_summary"].iterrows():
        for metric in METRICS:
            mean_col = f"{metric}_mean"
            if mean_col in row:
                combined_rows.append(
                    {
                        "method": row["model_label"],
                        "metric": metric,
                        "value": row[mean_col],
                        "family": "repr_learn_linear_probe",
                    }
                )

if not repr_data["dl_summary"].empty:
    for _, row in repr_data["dl_summary"].iterrows():
        for metric in METRICS:
            if metric in row:
                combined_rows.append(
                    {
                        "method": row["model_label"],
                        "metric": metric,
                        "value": row[metric],
                        "family": "repr_learn_dl_head",
                    }
                )

if not kintsugi["summary"].empty:
    for _, row in kintsugi["summary"].iterrows():
        for metric in METRICS:
            if metric in row:
                combined_rows.append(
                    {
                        "method": row["model"],
                        "metric": metric,
                        "value": row[metric],
                        "family": "kintsugi_dam",
                    }
                )

combined = pd.DataFrame(combined_rows)
if combined.empty:
    print("No metrics available for combined overview.")
else:
    combined = combined.dropna(subset=["value"]).copy()
    combined["metric_label"] = combined["metric"].map(METRIC_LABELS)

    family_titles = {
        "repr_learn_linear_probe": "Linear probe (GroupKFold mean)",
        "repr_learn_dl_head": "Frozen encoder + DL head",
        "kintsugi_dam": "Kintsugi DAM Whisper",
    }
    families = list(dict.fromkeys(combined["family"].tolist()))
    n_families = len(families)

    fig, axes = plt.subplots(
        n_families,
        1,
        figsize=(12, max(4.0, 3.5 * n_families)),
        squeeze=False,
    )
    for ax, family in zip(axes.ravel(), families):
        sub = combined[combined["family"] == family]
        if sub.empty:
            ax.set_visible(False)
            continue
        sns.barplot(data=sub, x="method", y="value", hue="metric_label", ax=ax)
        ax.set_ylim(0, 1.05)
        ax.set_title(family_titles.get(family, family))
        ax.set_xlabel("")
        ax.set_ylabel("Score")
        ax.tick_params(axis="x", rotation=25)
        ax.legend(title="", fontsize=9, loc="upper right")
    fig.suptitle("All audio-model metrics (repr learning + Kintsugi)", y=1.01)
    fig.tight_layout()
    save_fig(fig, "05_combined_overview_by_family.png")

    # Single-panel ROC-AUC comparison across everything
    roc = combined[combined["metric"] == "roc_auc"].copy()
    if not roc.empty:
        roc = roc.sort_values("value", ascending=True)
        fig_h = max(4.0, 0.55 * len(roc))
        fig, ax = plt.subplots(figsize=(10, fig_h))
        sns.barplot(data=roc, x="value", y="method", hue="family", dodge=False, ax=ax)
        x_left, x_right = ax.get_xlim()
        if x_left < x_right:
            ax.set_xlim(max(0, x_left), max(x_right, 1.05))
        ax.set_title("ROC-AUC comparison — all methods")
        ax.set_xlabel("ROC-AUC")
        ax.legend(title="Family", bbox_to_anchor=(1.02, 1), loc="upper left")
        save_fig(fig, "05_combined_roc_auc_comparison.png")

print(f"\nAll figures saved under: {FIG_DIR.resolve()}")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\05_combined_overview_by_family.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\05_combined_roc_auc_comparison.png

All figures saved under: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi


## RADAR figures

Linear-probe CSVs: results/metrics/repr_learn/radar_{KCL,CIBER,IISPV}_{wav2vec2,hubert,whisper}_{summary,cv_folds}.csv

Kintsugi (supervisor): data/processed/Nick/Nick_kinstugi_Health/RADAR/ (falls back to data/processed/Nick_kinstugi_Health/RADAR)

Run the setup cell first (save_fig, METRICS, seaborn theme). These cells do not mix Androids into the RADAR plots.

In [8]:
# --- RADAR loaders (repr_learn metrics + Nick Kintsugi summaries) ---
from sklearn.metrics import confusion_matrix

try:
    BASE, REPR_DIR, FIG_DIR, METRICS, METRIC_LABELS, save_fig, cv_summary_long, point_summary_long
except NameError as e:
    raise RuntimeError("Run the first setup cell first.") from e

KINTSUGI_RADAR_CANDIDATES = [
    BASE / "data" / "processed" / "Nick" / "Nick_kinstugi_Health" / "RADAR",
    BASE / "data" / "processed" / "Nick_kinstugi_Health" / "RADAR",
    BASE / "results" / "metrics" / "kintsugi_health" / "RADAR",
]
KINTSUGI_RADAR_DIR = next((p for p in KINTSUGI_RADAR_CANDIDATES if p.is_dir()), KINTSUGI_RADAR_CANDIDATES[0])

ENCODER_SUFFIXES = ("wav2vec2", "hubert", "whisper")


def _parse_radar_repr_stem(stem: str, suffix: str) -> tuple[str, str] | None:
    base = stem[: -len(suffix)]
    if not base.startswith("radar_"):
        return None
    rest = base[len("radar_") :]
    for encoder in ENCODER_SUFFIXES:
        token = f"_{encoder}"
        if rest.endswith(token):
            site = rest[: -len(token)]
            if site:
                return site, encoder
    return None


def load_radar_repr(metrics_dir: Path) -> dict[str, pd.DataFrame]:
    cv_summary_rows: list[pd.DataFrame] = []
    cv_fold_frames: list[pd.DataFrame] = []

    for path in sorted(metrics_dir.glob("radar_*.csv")):
        stem = path.stem
        if stem.endswith("_cv_folds"):
            parsed = _parse_radar_repr_stem(stem, "_cv_folds")
            kind = "folds"
        elif stem.endswith("_summary") and "_embeddings" not in stem:
            parsed = _parse_radar_repr_stem(stem, "_summary")
            kind = "summary"
        else:
            continue
        if parsed is None:
            continue
        site, encoder = parsed
        df = pd.read_csv(path)
        df["dataset"] = "RADAR"
        df["site"] = site
        df["backbone"] = encoder
        df["approach"] = "linear_probe_cv"
        df["model_label"] = f"RADAR {site} | {encoder} (linear probe)"
        if kind == "folds":
            cv_fold_frames.append(df)
        else:
            cv_summary_rows.append(df)

    return {
        "cv_summary": pd.concat(cv_summary_rows, ignore_index=True) if cv_summary_rows else pd.DataFrame(),
        "cv_folds": pd.concat(cv_fold_frames, ignore_index=True) if cv_fold_frames else pd.DataFrame(),
    }


def load_radar_kintsugi(metrics_dir: Path) -> dict[str, pd.DataFrame]:
    summary_rows: list[pd.DataFrame] = []
    pred_frames: list[pd.DataFrame] = []
    if not metrics_dir.is_dir():
        print(f"Warning: missing Kintsugi RADAR dir {metrics_dir}")
        return {"summary": pd.DataFrame(), "predictions": pd.DataFrame()}

    for path in sorted(metrics_dir.glob("dam_whisper_radar_*_summary.csv")):
        site = path.stem.replace("dam_whisper_radar_", "").replace("_summary", "")
        df = pd.read_csv(path)
        df["site"] = df["sites"] if "sites" in df.columns else site
        df["dataset"] = "RADAR"
        df["approach"] = "dam_whisper"
        df["model_label"] = df["model"] if "model" in df.columns else f"DAM-like Whisper (RADAR - {site})"
        summary_rows.append(df)

    for path in sorted(metrics_dir.glob("dam_whisper_radar_*_test_predictions.csv")):
        site = path.stem.replace("dam_whisper_radar_", "").replace("_test_predictions", "")
        df = pd.read_csv(path)
        df["site_group"] = site
        pred_frames.append(df)

    return {
        "summary": pd.concat(summary_rows, ignore_index=True) if summary_rows else pd.DataFrame(),
        "predictions": pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame(),
    }


radar_repr = load_radar_repr(REPR_DIR)
radar_kintsugi = load_radar_kintsugi(KINTSUGI_RADAR_DIR)

print("RADAR representation learning:")
for key, df in radar_repr.items():
    labels = df["model_label"].unique().tolist() if "model_label" in df.columns and not df.empty else []
    print(f"  {key}: {df.shape} — {labels}")

print(f"\nKintsugi RADAR dir: {KINTSUGI_RADAR_DIR}")
print("Kintsugi Health RADAR:")
for key, df in radar_kintsugi.items():
    print(f"  {key}: {df.shape}")

RADAR representation learning:
  cv_summary: (9, 14) — ['RADAR CIBER | hubert (linear probe)', 'RADAR CIBER | wav2vec2 (linear probe)', 'RADAR CIBER | whisper (linear probe)', 'RADAR IISPV | hubert (linear probe)', 'RADAR IISPV | wav2vec2 (linear probe)', 'RADAR IISPV | whisper (linear probe)', 'RADAR KCL | hubert (linear probe)', 'RADAR KCL | wav2vec2 (linear probe)', 'RADAR KCL | whisper (linear probe)']
  cv_folds: (45, 12) — ['RADAR CIBER | hubert (linear probe)', 'RADAR CIBER | wav2vec2 (linear probe)', 'RADAR CIBER | whisper (linear probe)', 'RADAR IISPV | hubert (linear probe)', 'RADAR IISPV | wav2vec2 (linear probe)', 'RADAR IISPV | whisper (linear probe)', 'RADAR KCL | hubert (linear probe)', 'RADAR KCL | wav2vec2 (linear probe)', 'RADAR KCL | whisper (linear probe)']

Kintsugi RADAR dir: C:\Users\janku\Documents\KCL\Research Project\Research Project\data\processed\Nick\Nick_kinstugi_Health\RADAR
Kintsugi Health RADAR:
  summary: (3, 17)
  predictions: (2958, 9)


In [ ]:
# --- 6. RADAR representation learning: GroupKFold linear-probe summary ---
cv_sum = radar_repr["cv_summary"]
if cv_sum.empty:
    print("No RADAR linear-probe CV summary files found.")
else:
    long = cv_summary_long(cv_sum)
    if "site" not in long.columns or long["site"].isna().all():
        long["site"] = long["model_label"].str.extract(r"RADAR ([^|]+) \|")[0].str.strip()

    g = sns.catplot(
        data=long,
        kind="bar",
        x="backbone",
        y="value",
        hue="metric",
        col="site",
        col_wrap=3,
        height=4.5,
        aspect=1.1,
        sharey=False,
    )
    g.set_axis_labels("Backbone", "Score")
    g.set_titles("{col_name}")
    for ax in g.axes.ravel():
        ax.set_ylim(0, 1.05)
    g.fig.suptitle("RADAR representation learning — linear probe (GroupKFold mean)", y=1.05)
    save_fig(g.fig, "06_radar_repr_linear_probe_cv_summary.png")

    sites = list(dict.fromkeys(long["site"].dropna().tolist()))
    fig, axes = plt.subplots(len(sites), len(METRICS), figsize=(5 * len(METRICS), 4.2 * max(len(sites), 1)), squeeze=False)
    for row_i, site in enumerate(sites):
        site_df = long[long["site"] == site]
        for col_i, metric in enumerate(METRICS):
            ax = axes[row_i, col_i]
            sub = site_df[site_df["metric"] == metric].copy()
            order = [e for e in ENCODER_SUFFIXES if e in set(sub["backbone"])]
            colors = sns.color_palette("Set2", len(order))
            for i, backbone in enumerate(order):
                row = sub[sub["backbone"] == backbone].iloc[0]
                ax.bar(i, row["value"], yerr=row["std"], capsize=4, color=colors[i], label=backbone)
            ax.set_xticks(range(len(order)))
            ax.set_xticklabels(order, rotation=20)
            ax.set_ylim(0, 1.05)
            ax.set_title(f"{site} — {METRIC_LABELS[metric]}")
            ax.set_ylabel("Score")
    fig.suptitle("RADAR linear probe CV — mean ± std by site and backbone", y=1.02)
    fig.tight_layout()
    save_fig(fig, "06_radar_repr_linear_probe_cv_summary_errbars.png")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\06_radar_repr_linear_probe_cv_summary.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\06_radar_repr_linear_probe_cv_summary_errbars.png


In [10]:
# --- 7. RADAR representation learning: per-fold CV curves + heatmaps ---
cv_folds = radar_repr["cv_folds"]
if cv_folds.empty:
    print("No RADAR CV fold files found.")
else:
    plot_metrics = [m for m in METRICS if m in cv_folds.columns]
    if plot_metrics:
        fig, axes = plt.subplots(1, len(plot_metrics), figsize=(5.4 * len(plot_metrics), 5), squeeze=False)
        for ax, metric in zip(axes.ravel(), plot_metrics):
            sns.lineplot(
                data=cv_folds,
                x="fold",
                y=metric,
                hue="backbone",
                style="site",
                marker="o",
                ax=ax,
            )
            ax.set_title(f"RADAR CV {METRIC_LABELS[metric]} by fold")
            ax.set_xlabel("Fold")
            ax.set_xticks(sorted(cv_folds["fold"].dropna().unique()))
            ax.set_ylim(0, 1.05)
            ax.legend(title="", fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
        fig.tight_layout()
        save_fig(fig, "07_radar_repr_linear_probe_cv_folds_lineplots.png")

    for site, site_df in cv_folds.groupby("site"):
        for metric in plot_metrics:
            pivot = site_df.pivot_table(index="backbone", columns="fold", values=metric, aggfunc="first")
            if pivot.empty:
                continue
            fig, ax = plt.subplots(figsize=(8, max(3.5, 0.7 * len(pivot))))
            sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis", vmin=0, vmax=1, ax=ax)
            ax.set_title(f"RADAR {site} linear probe CV — {METRIC_LABELS[metric]}")
            save_fig(fig, f"07_radar_repr_linear_probe_cv_heatmap_{site}_{metric}.png")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\07_radar_repr_linear_probe_cv_folds_lineplots.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\07_radar_repr_linear_probe_cv_heatmap_CIBER_accuracy.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\07_radar_repr_linear_probe_cv_heatmap_CIBER_f1.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\07_radar_repr_linear_probe_cv_heatmap_CIBER_roc_auc.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\07_radar_repr_linear_probe_cv_heatmap_IISPV_accuracy.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\07_radar_repr_linear_probe_cv_heatmap_IISPV_f1.png
Saved: C:\Users\janku\Do

In [11]:
# --- 8. RADAR Kintsugi Health: DAM-like Whisper (supervisor held-out test) ---
k_sum = radar_kintsugi["summary"]
k_preds = radar_kintsugi["predictions"]

if k_sum.empty:
    print("No RADAR Kintsugi summary found.")
else:
    long = point_summary_long(k_sum, model_col="model_label")
    fig, ax = plt.subplots(figsize=(11, 5.5))
    sns.barplot(data=long, x="site", y="value", hue="metric", ax=ax)
    ax.set_ylim(0, 1.05)
    ax.set_title("RADAR Kintsugi Health — DAM-like Whisper (held-out test)")
    ax.set_xlabel("Site group")
    ax.set_ylabel("Score")
    ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_fig(fig, "08_radar_kintsugi_dam_test_summary.png")

if not k_preds.empty and "pred_prob" in k_preds.columns:
    hue_col = "subgroup" if "subgroup" in k_preds.columns else "site_group"
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(
        data=k_preds,
        x="pred_prob",
        hue=hue_col,
        multiple="stack",
        bins=25,
        ax=ax,
    )
    ax.set_title("RADAR Kintsugi DAM — P(depressed) by site")
    ax.set_xlabel("P(depressed)")
    save_fig(fig, "08_radar_kintsugi_dam_pred_prob_by_site.png")

    if {"depressed", "pred_label", "site_group"}.issubset(k_preds.columns):
        groups = list(dict.fromkeys(k_preds["site_group"].tolist()))
        fig, axes = plt.subplots(1, len(groups), figsize=(4.8 * len(groups), 4.2), squeeze=False)
        for ax, site in zip(axes.ravel(), groups):
            sub = k_preds[k_preds["site_group"] == site]
            cm = pd.crosstab(sub["depressed"], sub["pred_label"])
            cm = cm.reindex(index=[0, 1], columns=[0, 1], fill_value=0)
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True (depressed)")
            ax.set_title(site)
        fig.suptitle("RADAR Kintsugi DAM — confusion matrices (test)", y=1.03)
        fig.tight_layout()
        save_fig(fig, "08_radar_kintsugi_dam_confusion_matrix.png")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\08_radar_kintsugi_dam_test_summary.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\08_radar_kintsugi_dam_pred_prob_by_site.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\08_radar_kintsugi_dam_confusion_matrix.png


In [12]:
# --- 9. RADAR combined overview: linear probes vs Kintsugi DAM ---
combined_rows = []

if not radar_repr["cv_summary"].empty:
    for _, row in radar_repr["cv_summary"].iterrows():
        for metric in METRICS:
            mean_col = f"{metric}_mean"
            if mean_col in row:
                combined_rows.append(
                    {
                        "method": row["model_label"],
                        "site": row["site"],
                        "metric": metric,
                        "value": row[mean_col],
                        "family": "repr_learn_linear_probe",
                    }
                )

if not radar_kintsugi["summary"].empty:
    for _, row in radar_kintsugi["summary"].iterrows():
        for metric in METRICS:
            if metric in row:
                combined_rows.append(
                    {
                        "method": row["model_label"],
                        "site": row.get("site", row.get("sites")),
                        "metric": metric,
                        "value": row[metric],
                        "family": "kintsugi_dam",
                    }
                )

combined = pd.DataFrame(combined_rows)
if combined.empty:
    print("No RADAR metrics available for combined overview.")
else:
    combined = combined.dropna(subset=["value"]).copy()
    combined["metric_label"] = combined["metric"].map(METRIC_LABELS)

    family_titles = {
        "repr_learn_linear_probe": "RADAR linear probe (GroupKFold mean)",
        "kintsugi_dam": "RADAR Kintsugi DAM Whisper",
    }
    families = list(dict.fromkeys(combined["family"].tolist()))
    fig, axes = plt.subplots(
        len(families),
        1,
        figsize=(13, max(4.2, 3.8 * len(families))),
        squeeze=False,
    )
    for ax, family in zip(axes.ravel(), families):
        sub = combined[combined["family"] == family]
        sns.barplot(data=sub, x="method", y="value", hue="metric_label", ax=ax)
        ax.set_ylim(0, 1.05)
        ax.set_title(family_titles.get(family, family))
        ax.set_xlabel("")
        ax.set_ylabel("Score")
        ax.tick_params(axis="x", rotation=25)
        ax.legend(title="", fontsize=9, loc="upper right")
    fig.suptitle("RADAR audio-model metrics (repr learning + Kintsugi)", y=1.01)
    fig.tight_layout()
    save_fig(fig, "09_radar_combined_overview_by_family.png")

    roc = combined[combined["metric"] == "roc_auc"].copy()
    if not roc.empty:
        roc = roc.sort_values("value", ascending=True)
        fig, ax = plt.subplots(figsize=(10, max(4.0, 0.45 * len(roc))))
        sns.barplot(data=roc, x="value", y="method", hue="family", dodge=False, ax=ax)
        ax.set_xlim(0, 1.05)
        ax.set_title("RADAR ROC-AUC — linear probes vs Kintsugi")
        ax.set_xlabel("ROC-AUC")
        ax.legend(title="Family", bbox_to_anchor=(1.02, 1), loc="upper left")
        save_fig(fig, "09_radar_combined_roc_auc_comparison.png")

print(f"\nRADAR figures saved under: {FIG_DIR.resolve()}")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\09_radar_combined_overview_by_family.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi\09_radar_combined_roc_auc_comparison.png

RADAR figures saved under: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\reprlearn_and_kintsugi
